> 在线讲义：[第 8 章 · 数 FLOPs：训练 ≈ 6 × 参数 × token 数从哪来，注意力什么时候不能忽略](https://wikieden.github.io/beyond-transformer/lessons/08-flops.html) · 本书《超越 Transformer》

# 第 8 章 · 数 FLOPs：训练 ≈ 6 × 参数 × token 数从哪来，注意力什么时候不能忽略

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/08-flops.html) · 仓库里 `beyond-transformer/notes/lessons/08-flops.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 数一遍 | 2mnk | `count_matmul()`、`matmul_flops(2, 3, 4)` | 48 |
| 第二步 6ND | 2N、6N | `toy_summary()` | 参数 162,304，前向 390,144 |
| 第三步 一层的构成 | 各部分 | `layer_forward_per_token(64, 128, 256)` | 合计 131,072 |
| 第四步 注意力占比 | n/(6d) | `attn_ratio(n, d)` | 0.083、1.333 |

注意力的 QKᵀ 和 AV 不对应参数，每层每个 token 多 4·n·d 次；它和 24·d² 的比值是 n / (6d)。
只算一个玩具配置；给定配置算参数量、FLOPs、KV cache 并对官方数的 sizing.py 留给 T-M1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `matmul_flops`**：(m × k) @ (k × n)：输出 m·n 个数，每个数 k 次乘、k 次加。

In [ ]:
def matmul_flops(m, n, k):
    """(m × k) @ (k × n)：输出 m·n 个数，每个数 k 次乘、k 次加。"""
    return 2 * m * n * k

**函数 `count_matmul`**：真的数一遍：乘法和加法各多少次。

In [ ]:
def count_matmul(m=2, n=3, k=4):
    """真的数一遍：乘法和加法各多少次。"""
    a, b = np.ones((m, k)), np.ones((k, n))
    mul = add = 0
    out = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            s = 0.0
            for t in range(k):
                s += a[i, t] * b[t, j]
                mul += 1
                add += 1
            out[i, j] = s
    return mul, add

In [ ]:
TOY = dict(d=64, L=2, n=128, V=1000, f=256)   # 宽 64、2 层、序列 128、词表 1000、FFN 中间 256

**函数 `layer_forward_per_token`**：一层、一个 token 的前向 FLOPs（不含 softmax 等逐元素运算）。

In [ ]:
def layer_forward_per_token(d, n, f):
    """一层、一个 token 的前向 FLOPs（不含 softmax 等逐元素运算）。"""
    return {
        "Q、K、V 投影": 2 * 3 * d * d,
        "输出投影": 2 * d * d,
        "FFN 两层": 2 * 2 * d * f,
        "注意力 QKᵀ": 2 * n * d,
        "注意力 AV": 2 * n * d,
    }

**函数 `toy_summary`**：小工具。

In [ ]:
def toy_summary(d=64, L=2, n=128, V=1000, f=256):
    per = layer_forward_per_token(d, n, f)
    params_layer = 4 * d * d + 2 * d * f
    params = L * params_layer + V * d          # 嵌入和输出层共享
    fwd = L * sum(per.values()) + 2 * V * d    # 最后乘回词表
    attn = L * (per["注意力 QKᵀ"] + per["注意力 AV"])
    return {"参数": params, "前向/token": fwd, "2×参数": 2 * params, "注意力部分": attn, "比例": attn / fwd}

**函数 `attn_ratio`**：f = 4d 时：每层参数矩阵 12d²，前向 24d²；注意力 4nd。比值 = n / (6d)。

In [ ]:
def attn_ratio(n, d):
    """f = 4d 时：每层参数矩阵 12d²，前向 24d²；注意力 4nd。比值 = n / (6d)。"""
    return 4 * n * d / (24 * d * d)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 一次矩阵乘法")
mul, add = count_matmul()
print(f"  (2 × 4) @ (4 × 3)：乘 {mul} 次、加 {add} 次，合计 {mul + add} = 2·2·3·4 = {matmul_flops(2, 3, 4)}")
print("\n第二步 一层一个 token 的前向（d = 64、n = 128、f = 256）")
per = layer_forward_per_token(64, 128, 256)
for k, v in per.items():
    print(f"  {k}：{v:,}")
print(f"  合计 {sum(per.values()):,}")

In [ ]:
print("\n第三步 前向 ≈ 2 × 参数，训练 ≈ 6 × 参数 × token")
s = toy_summary()
print(f"  玩具模型参数 {s['参数']:,}；每个 token 前向 {s['前向/token']:,} FLOPs，2 × 参数 = {s['2×参数']:,}")
print(f"  训练 100 万 token ≈ 6 × {s['参数']:,} × 1e6 = {6 * s['参数'] * 1e6:.3e} FLOPs")

In [ ]:
print("\n第四步 注意力什么时候不能忽略：比值 n / (6d)")
print(f"  玩具模型（n = 128、d = 64）：注意力占前向 {s['比例']:.1%}")
for n, d in ((2048, 4096), (32768, 4096), (128, 64)):
    print(f"  n = {n:6d}、d = {d}：注意力 / 参数矩阵 = {attn_ratio(n, d):.3f}")

## 3. 代码里才看得到的

- **嵌入层的“前向”几乎不花 FLOPs**：查表只是取一行，不是矩阵乘法；但输出层（乘回词表）是实打实的 2·V·d。嵌入和输出共享权重时，6ND 里的那部分参数其实只在输出层“花钱”。
- **只算了一个玩具配置**：对真实模型（GPT-2、Llama、Qwen）算参数量和 FLOPs、和官方数对齐的 sizing.py 留给里程碑 T-M1。

## 4. 自己改一改

- 把 `toy_summary` 的 n 改成 1024，注意力的占比变成多少？
- 把 f 改成 3d（SwiGLU 常用的 8/3 d 附近），每层参数和 n/(6d) 的公式要怎么改？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 矩阵乘法 2mnk，一个参数一个 token 两次运算。
> 反向两回同样大：一回传给上一层，一回交给优化器；前二后四，训练六 N D。
> QK 和 AV 没参数，随序列长度涨。
> 比一比 n 和六倍宽：远小于就忽略，接近了就算上。